# Component-Level Testing

Deep-dive into individual components with comprehensive train-test evaluation.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from featransform.utils.data_generator import DatasetGenerator
from featransform.processing.imputation import CoreImputer
from featransform.processing.encoding import CategoricalEncoder
from featransform.processing.datetime_features import DatetimeExpander
from featransform.processing.scaling import FeatureScaler
from featransform.features.anomaly import AnomalyEnsemble, IsolationForestStrategy
from featransform.features.clustering import ClusteringEnsemble, KMeansStrategy
from featransform.features.dimensionality import DimensionalityEnsemble, PCAStrategy
from featransform.optimization.selector import FeatureSelector
from featransform.core.enums import ImputationStrategy, EncodingStrategy, ScalingStrategy, SelectionStrategy

# Identifier / index columns: they pass through to the output but must not act as
# imputation predictors or feed the distance / variance based ensembles.
ID_COLUMNS = ["id"]

## Generate Dataset

In [ ]:
# Generate a sample dataset — for demonstration only.
# When using Featransform for real, swap this for your own data (X) and target (y).
X_full, y_full = DatasetGenerator.complex_dataset(n_samples=10000, task='binary_classification')
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(X_full, y_full, test_size=0.3, random_state=42)

## Encoding

### Datetime expansion

Datetime columns cannot be consumed by the scikit-learn estimators behind the anomaly, clustering and dimensionality components, so expand them into calendar parts and cyclic encodings first. The full `Featransform` pipeline does this automatically via `ProcessingConfig.handle_datetime`; at component level you call it directly.

In [ ]:
dt_expander = DatetimeExpander()
dt_expander.fit(X_train_full)
X_train_full = dt_expander.transform(X_train_full)
X_test_full = dt_expander.transform(X_test_full)
print(f"Expanded datetime columns: {dt_expander.get_feature_names()}")
print(f"Shape after expansion: {X_train_full.shape}")

In [ ]:
encoder = CategoricalEncoder(strategy=EncodingStrategy.LABEL)
encoder.fit(X_train_full, y_train_full)
X_train_proc = encoder.transform(X_train_full)
X_test_proc = encoder.transform(X_test_full)

## Imputation

In [ ]:
imputer = CoreImputer(strategy=ImputationStrategy.ITERATIVE, exclude_columns=ID_COLUMNS)
imputer.fit(X_train_proc)
X_train_proc = imputer.transform(X_train_proc)
X_test_proc = imputer.transform(X_test_proc)

## Feature Engineering

In [ ]:
# The ensembles are distance / variance based, so they get a dedicated input:
# identifier columns removed and every column standardized. The processed frame
# itself stays in its native scale and is what the engineered features are
# concatenated onto. The full Featransform pipeline does both automatically.
scaler = FeatureScaler(strategy=ScalingStrategy.STANDARD)
X_train_fe = scaler.fit(X_train_proc.drop(columns=ID_COLUMNS)).transform(X_train_proc.drop(columns=ID_COLUMNS))
X_test_fe = scaler.transform(X_test_proc.drop(columns=ID_COLUMNS))

anomaly_ens = AnomalyEnsemble(strategies=[IsolationForestStrategy(n_estimators=100, contamination=0.1)], verbose=False)
anomaly_ens.fit(X_train_fe)
train_anom = anomaly_ens.transform(X_train_fe).features
test_anom = anomaly_ens.transform(X_test_fe).features

cluster_ens = ClusteringEnsemble(strategies=[KMeansStrategy(n_clusters=3, random_state=42)], verbose=False)
cluster_ens.fit(X_train_fe)
train_clust = cluster_ens.transform(X_train_fe).features
test_clust = cluster_ens.transform(X_test_fe).features

dim_ens = DimensionalityEnsemble(strategies=[PCAStrategy(n_components=5)], verbose=False)
dim_ens.fit(X_train_fe)
train_dim = dim_ens.transform(X_train_fe).features
test_dim = dim_ens.transform(X_test_fe).features

## Combine Features

In [ ]:
X_train_all = pd.concat([X_train_proc, train_anom, train_clust, train_dim], axis=1)
X_test_all = pd.concat([X_test_proc, test_anom, test_clust, test_dim], axis=1)

## Feature Selection

In [ ]:
selector = FeatureSelector(strategy=SelectionStrategy.IMPORTANCE, min_features=10, verbose=False)
selector.fit(X_train_all, y_train_full)
selected_features = selector.select_by_threshold(0.7)
selector.set_selected_features(selected_features)
X_train_final = selector.transform(X_train_all)
X_test_final = selector.transform(X_test_all)

scores = selector.get_feature_scores()
if scores:
    top_10 = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:10]
    for feat, score in top_10:
        print(f"{feat}: {score:.4f}")